# 03 · 参数拟合与模型验证

本 Notebook 用于：
1. 参数拟合（lmfit 最小化 RMSE）
2. Bootstrap 参数置信区间
3. 模型验证（RMSE / MAPE / R²）
4. 输出验证图（模拟 vs FluNet 观测）

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from model.params import ModelParams
from model.solver import solve_seiqr
from calibration.fitting import fit_model, prepare_obs_timeseries
from calibration.bootstrap import bootstrap_params, bootstrap_trajectory
from calibration.validation import compute_metrics
from plots._style import setup_style
from plots.epidemic_curve import plot_epidemic_curve

setup_style()

DATA_DIR = Path('../data/processed')
FIG_DIR  = Path('../output/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. 加载参数和观测数据

In [ ]:
p_init = ModelParams.from_yaml('../config.yaml') if Path('../config.yaml').exists() else ModelParams()

weekly_path = DATA_DIR / 'weekly_positivity.csv'
if not weekly_path.exists():
    print('请先运行: python data/fetch_flunet.py')
else:
    obs_weekly = pd.read_csv(weekly_path, parse_dates=['date'])
    obs_df = prepare_obs_timeseries(obs_weekly, p_init)
    print(f'观测数据点: {len(obs_df)}')
    print(f'时间范围: 第 {obs_df["t_sim"].min():.0f} 天 → 第 {obs_df["t_sim"].max():.0f} 天')
    obs_df.head()

## 2. 参数拟合

In [ ]:
result = fit_model(obs_df, p_init, method='leastsq', max_nfev=1000)

print(f'拟合状态: {"成功" if result.success else "未收敛"}')
print(f'RMSE = {result.rmse:.6f}')
print(f'R²   = {result.r_squared:.4f}')
print(f'观测点数: {result.n_obs}')
print('\n最优参数:')
for k, v in result.params_best.items():
    init_v = getattr(p_init, k, None)
    print(f'  {k}: {v:.4f}  (初始: {init_v:.4f})')

In [ ]:
# 更新参数，重新求解
p_fit = p_init.update(**result.params_best)
df_fit = solve_seiqr(p_fit)

plot_epidemic_curve(
    df_fit, obs_df=obs_df,
    title='参数拟合结果：模拟 vs WHO FluNet 观测',
    output_path=FIG_DIR / '03_fitting_result.png',
    show=True
)

## 3. Bootstrap 置信区间

In [ ]:
ci_result = bootstrap_params(obs_df, p_fit, n_bootstrap=200, random_seed=42)

print('参数 95% 置信区间:')
for name, ci in ci_result.get('param_ci', {}).items():
    print(f'  {name}: [{ci["lo"]:.4f}, {ci["hi"]:.4f}]  mean={ci["mean"]:.4f} ± {ci["std"]:.4f}')

print(f'\nBootstrap 收敛率: {ci_result["success_rate"]:.1%}')

In [ ]:
# Bootstrap 轨迹置信带
traj = bootstrap_trajectory(obs_df, p_fit, n_bootstrap=100, random_seed=42)

if traj:
    plot_epidemic_curve(
        df_fit, obs_df=obs_df, ci_band=traj,
        title='Bootstrap 95% 置信带',
        output_path=FIG_DIR / '03_bootstrap_ci.png',
        show=True
    )

## 4. 模型验证指标

In [ ]:
metrics = compute_metrics(df_fit, obs_df)

print('模型验证指标:')
print(f'  RMSE:      {metrics.rmse:.6f}')
print(f'  MAE:       {metrics.mae:.6f}')
print(f'  MAPE:      {metrics.mape:.2f}%')
print(f'  R²:        {metrics.r_squared:.4f}')
print(f'  Pearson r: {metrics.pearson_r:.4f}')
print(f'  峰值时间误差: {metrics.peak_error_days:.1f} 天')
print(f'  峰值率误差:  {metrics.peak_rate_err:.4f}')